In [0]:
RAW_PATH = '/Volumes/workspace/default/ecommerce/'
BRONZE_PATH='/Volumes/workspace/default/ecommerce/bronze1/'
SILVER_PATH='/Volumes/workspace/default/ecommerce/silver1/'
GOLD_PATH='/Volumes/workspace/default/ecommerce/gold/'



In [0]:
txn = spark.read.format('Delta').load(BRONZE_PATH + 'transactions')
cust = spark.read.format('Delta').load(BRONZE_PATH + 'customers')
prod = spark.read.format('Delta').load(BRONZE_PATH + 'products')
store = spark.read.format('Delta').load(BRONZE_PATH + 'stores')
promo = spark.read.format('Delta').load(BRONZE_PATH + 'promotions')
fb = spark.read.format('Delta').load (BRONZE_PATH + 'feedbacks')



In [0]:
display(txn)

transaction_id,customer_id,product_id,store_id,transaction_date,quantity,total_amount
1,9,1,4,2022-12-13,3,586.05
2,7,1,2,2023-07-07,2,390.7
3,9,7,3,2024-02-29,3,1036.65
4,15,2,2,2024-01-22,5,482.4
5,12,1,4,2023-12-26,5,976.75
6,9,2,3,2024-08-23,4,385.92
7,5,2,2,2022-03-30,1,96.48
8,9,4,5,2022-10-28,1,427.22
9,11,2,4,2023-12-26,5,482.4
10,14,2,1,2022-06-21,3,289.44


In [0]:
from pyspark.sql.functions import col, to_date,coalesce,lit

txn_clean = (
    txn.dropDuplicates(["transaction_id"])
       .withColumn(
           "transaction_date",
           to_date(col("transaction_date"), "dd-MM-yyyy")
       )
       .withColumn("quantity", col("quantity").cast("int"))
       .withColumn("total_amount", col("total_amount").cast("double"))\
           .withColumn("total_amount",coalesce(col("total_amount"),lit(0.0))
           )
)
display(txn_clean)

transaction_id,customer_id,product_id,store_id,transaction_date,quantity,total_amount
1,9,1,4,2022-12-13,3,586.05
2,7,1,2,2023-07-07,2,390.7
3,9,7,3,2024-02-29,3,1036.65
4,15,2,2,2024-01-22,5,482.4
5,12,1,4,2023-12-26,5,976.75
6,9,2,3,2024-08-23,4,385.92
7,5,2,2,2022-03-30,1,96.48
8,9,4,5,2022-10-28,1,427.22
9,11,2,4,2023-12-26,5,482.4
10,14,2,1,2022-06-21,3,289.44


In [0]:
silver = (txn_clean.alias("t")
          .join(cust.alias("c"),"customer_id","left")
          .join(prod.alias("p"),"product_id","left")
          .join(store.alias("s"),"store_id","left")
          .join(promo.alias("pr"),"product_id","left")
          .join(fb.alias("f"),"customer_id","left")
          )

In [0]:
display(silver)

customer_id,product_id,store_id,transaction_id,transaction_date,quantity,total_amount,customer_name,city,signup_date,product_name,category,price,brand,store_name,location,manager,opened_date,promotion_id,discount,start_date,end_date,channel,feedback_id,product_id,rating,review,date
8,10,4,12,2022-10-10,5,505.2,Customer_8,Delhi,2022-08-09,Product_10,Electronics,101.04,Brand_2,Store_4,Kolkata,Manager_4,2018-02-25,null,null,null,null,null,8,4,5,Poor quality,2022-01-14
11,7,4,18,2024-02-16,2,691.1,Customer_11,Chennai,2022-11-11,Product_7,Electronics,345.55,Brand_5,Store_4,Kolkata,Manager_4,2018-02-25,10,23,2023-06-02,2023-06-12,Online,null,null,null,null,null
7,3,2,16,2024-06-11,3,221.88,Customer_7,Hyderabad,2022-07-22,Product_3,Clothing,73.96,Brand_3,Store_2,Bangalore,Manager_2,2018-03-13,3,8,2022-10-06,2022-11-20,Social Media,15,2,3,Value for money,2023-03-06
12,1,4,5,2023-12-26,5,976.75,Customer_12,Mumbai,2022-12-03,Product_1,Furniture,195.35,Brand_2,Store_4,Kolkata,Manager_4,2018-02-25,9,28,2023-07-26,2023-09-21,Social Media,14,10,1,Average experience,2023-07-02
14,2,1,10,2022-06-21,3,289.44,Customer_14,Hyderabad,2023-02-25,Product_2,Electronics,96.48,Brand_1,Store_1,Kolkata,Manager_1,2015-10-18,8,29,2022-09-26,2022-10-18,Online,9,2,2,Value for money,2023-12-16
9,1,4,1,2022-12-13,3,586.05,Customer_9,Kolkata,2022-09-14,Product_1,Furniture,195.35,Brand_2,Store_4,Kolkata,Manager_4,2018-02-25,9,28,2023-07-26,2023-09-21,Social Media,null,null,null,null,null
9,7,3,3,2024-02-29,3,1036.65,Customer_9,Kolkata,2022-09-14,Product_7,Electronics,345.55,Brand_5,Store_3,Delhi,Manager_3,2019-10-12,10,23,2023-06-02,2023-06-12,Online,null,null,null,null,null
11,2,4,20,2024-02-05,1,96.48,Customer_11,Chennai,2022-11-11,Product_2,Electronics,96.48,Brand_1,Store_4,Kolkata,Manager_4,2018-02-25,8,29,2022-09-26,2022-10-18,Online,null,null,null,null,null
7,1,2,2,2023-07-07,2,390.7,Customer_7,Hyderabad,2022-07-22,Product_1,Furniture,195.35,Brand_2,Store_2,Bangalore,Manager_2,2018-03-13,9,28,2023-07-26,2023-09-21,Social Media,15,2,3,Value for money,2023-03-06
10,4,2,13,2022-02-27,1,427.22,Customer_10,Pune,2022-10-01,Product_4,Furniture,427.22,Brand_2,Store_2,Bangalore,Manager_2,2018-03-13,null,null,null,null,null,10,10,3,Poor quality,2022-09-03


In [0]:
from pyspark.sql.functions import date_format, col, when, lit

silver = (
    silver
    .withColumn("transaction_date", date_format(col("transaction_date"), "yyyy-MM-dd"))
    .withColumn("is_valid_store", when(col("location").isNotNull(), lit(True)).otherwise(lit(False)))
    .withColumn("has_valid_customer", when(col("customer_name").isNotNull(), lit(True)).otherwise(lit(False)))
    .withColumn("promotion_active", when(col("discount").isNotNull(), lit(True)).otherwise(lit(False)))
    .withColumn(
        "final_amount",
        when(col("discount").isNotNull(), col("total_amount") - col("discount"))
        .otherwise(col("total_amount"))
    )
)

display(silver)

customer_id,product_id,store_id,transaction_id,transaction_date,quantity,total_amount,customer_name,city,signup_date,product_name,category,price,brand,store_name,location,manager,opened_date,promotion_id,discount,start_date,end_date,channel,feedback_id,product_id,rating,review,date,is_valid_store,has_valid_customer,promotion_active,final_amount
8,10,4,12,2022-10-10,5,505.2,Customer_8,Delhi,2022-08-09,Product_10,Electronics,101.04,Brand_2,Store_4,Kolkata,Manager_4,2018-02-25,null,null,null,null,null,8,4,5,Poor quality,2022-01-14,true,true,false,505.2
11,7,4,18,2024-02-16,2,691.1,Customer_11,Chennai,2022-11-11,Product_7,Electronics,345.55,Brand_5,Store_4,Kolkata,Manager_4,2018-02-25,10,23,2023-06-02,2023-06-12,Online,null,null,null,null,null,true,true,true,668.1
7,3,2,16,2024-06-11,3,221.88,Customer_7,Hyderabad,2022-07-22,Product_3,Clothing,73.96,Brand_3,Store_2,Bangalore,Manager_2,2018-03-13,3,8,2022-10-06,2022-11-20,Social Media,15,2,3,Value for money,2023-03-06,true,true,true,213.88
12,1,4,5,2023-12-26,5,976.75,Customer_12,Mumbai,2022-12-03,Product_1,Furniture,195.35,Brand_2,Store_4,Kolkata,Manager_4,2018-02-25,9,28,2023-07-26,2023-09-21,Social Media,14,10,1,Average experience,2023-07-02,true,true,true,948.75
14,2,1,10,2022-06-21,3,289.44,Customer_14,Hyderabad,2023-02-25,Product_2,Electronics,96.48,Brand_1,Store_1,Kolkata,Manager_1,2015-10-18,8,29,2022-09-26,2022-10-18,Online,9,2,2,Value for money,2023-12-16,true,true,true,260.44
9,1,4,1,2022-12-13,3,586.05,Customer_9,Kolkata,2022-09-14,Product_1,Furniture,195.35,Brand_2,Store_4,Kolkata,Manager_4,2018-02-25,9,28,2023-07-26,2023-09-21,Social Media,null,null,null,null,null,true,true,true,558.05
9,7,3,3,2024-02-29,3,1036.65,Customer_9,Kolkata,2022-09-14,Product_7,Electronics,345.55,Brand_5,Store_3,Delhi,Manager_3,2019-10-12,10,23,2023-06-02,2023-06-12,Online,null,null,null,null,null,true,true,true,1013.6500000000001
11,2,4,20,2024-02-05,1,96.48,Customer_11,Chennai,2022-11-11,Product_2,Electronics,96.48,Brand_1,Store_4,Kolkata,Manager_4,2018-02-25,8,29,2022-09-26,2022-10-18,Online,null,null,null,null,null,true,true,true,67.48
7,1,2,2,2023-07-07,2,390.7,Customer_7,Hyderabad,2022-07-22,Product_1,Furniture,195.35,Brand_2,Store_2,Bangalore,Manager_2,2018-03-13,9,28,2023-07-26,2023-09-21,Social Media,15,2,3,Value for money,2023-03-06,true,true,true,362.7
10,4,2,13,2022-02-27,1,427.22,Customer_10,Pune,2022-10-01,Product_4,Furniture,427.22,Brand_2,Store_2,Bangalore,Manager_2,2018-03-13,null,null,null,null,null,10,10,3,Poor quality,2022-09-03,true,true,false,427.22


In [0]:
# Check duplicate column names
from collections import Counter

print([c for c, n in Counter(silver.columns).items() if n > 1])

['product_id']


In [0]:
silver = silver.toDF(*[
    c if silver.columns.index(c) == i else f"{c}_duplicate"
    for i, c in enumerate(silver.columns)
])

In [0]:
silver.write.format("delta").mode("overwrite").save(SILVER_PATH + 'transactions_enriched')
